# Demographic-features comparison

Two rows (AUROC, AUPR) x three disease columns (Lupus, HIV, Covid19). In each panel, three TCR-based methods are shown as paired bars: without demographics vs. with demographics (age, sex, ancestry) added as input features. A horizontal dashed line marks the metric value achievable from demographics alone, i.e. the confounder-only baseline.

A Delta value above each pair quantifies how much adding demographics shifted the metric. Bars marked "N/A" denote runs not yet completed.

**Data sources**
- Demographics only: `evals/demographic_features_disease_classification.py`
- LogReg (V/J genes): standard V/J-only LR run
- Demo + LogReg (V/J): `evals/vjgene_demographics_disease_classification.py`
- DeepTCR: standard DeepTCR run
- Demo + DeepTCR: not yet implemented
- ABMIL: standard ABMIL run
- Demo + ABMIL: not yet implemented

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## Config

In [ ]:
DISEASES = ["Lupus", "HIV", "Covid19"]
METHODS = ["LogReg (V/J genes)", "DeepTCR", "ABMIL"]
METRICS = ["AUROC", "AUPR"]

NA = np.nan

# Visual params
COLOR_NO_DEMO   = "#9ecae1"
COLOR_WITH_DEMO = "#2171b5"
COLOR_DEMO_LINE = "#d62728"
COLOR_DELTA_POS = "#2ca02c"
COLOR_DELTA_NEG = "#d62728"

BAR_WIDTH = 0.36
Y_LO, Y_HI = 0.5, 1.05

## Results table

(without_demo, with_demo) per (disease, method, metric).

In [ ]:
RESULTS = {
    # --- Lupus ---
    ("Lupus",   "LogReg (V/J genes)", "AUROC"): (0.9172, 0.9376),
    ("Lupus",   "LogReg (V/J genes)", "AUPR"):  (0.8219, 0.8197),
    ("Lupus",   "DeepTCR",            "AUROC"): (NA, NA),
    ("Lupus",   "DeepTCR",            "AUPR"):  (NA, NA),
    ("Lupus",   "ABMIL",              "AUROC"): (NA, NA),
    ("Lupus",   "ABMIL",              "AUPR"):  (NA, NA),

    # --- HIV ---
    ("HIV",     "LogReg (V/J genes)", "AUROC"): (0.8882, 0.8385),
    ("HIV",     "LogReg (V/J genes)", "AUPR"):  (0.7238, 0.7826),
    ("HIV",     "DeepTCR",            "AUROC"): (NA, NA),
    ("HIV",     "DeepTCR",            "AUPR"):  (NA, NA),
    ("HIV",     "ABMIL",              "AUROC"): (NA, NA),
    ("HIV",     "ABMIL",              "AUPR"):  (NA, NA),

    # --- Covid19 ---
    ("Covid19", "LogReg (V/J genes)", "AUROC"): (0.9973, 0.9000),
    ("Covid19", "LogReg (V/J genes)", "AUPR"):  (0.9879, 0.8889),
    ("Covid19", "DeepTCR",            "AUROC"): (NA, NA),
    ("Covid19", "DeepTCR",            "AUPR"):  (NA, NA),
    ("Covid19", "ABMIL",              "AUROC"): (NA, NA),
    ("Covid19", "ABMIL",              "AUPR"):  (NA, NA),
}

# Demographics-only baseline per (disease, metric).
DEMO_ONLY = {
    ("Lupus",   "AUROC"): 0.7669,
    ("Lupus",   "AUPR"):  0.5876,
    ("HIV",     "AUROC"): 0.8694,
    ("HIV",     "AUPR"):  0.6691,
    ("Covid19", "AUROC"): 0.8634,
    ("Covid19", "AUPR"):  0.7018,
}

## Plot helpers

In [ ]:
def _draw_pair(ax, x_center, no_demo, with_demo):
    """Draw a (without-demo, with-demo) bar pair centered at x_center."""
    x_left  = x_center - BAR_WIDTH / 2
    x_right = x_center + BAR_WIDTH / 2

    if not np.isnan(no_demo):
        ax.bar(x_left, no_demo, BAR_WIDTH,
               color=COLOR_NO_DEMO, edgecolor="black", linewidth=0.6)
    else:
        ax.text(x_left, Y_LO + 0.02, "N/A",
                ha="center", va="bottom", fontsize=8, color="0.5",
                rotation=90)

    if not np.isnan(with_demo):
        ax.bar(x_right, with_demo, BAR_WIDTH,
               color=COLOR_WITH_DEMO, edgecolor="black", linewidth=0.6)
    else:
        ax.text(x_right, Y_LO + 0.02, "N/A",
                ha="center", va="bottom", fontsize=8, color="0.5",
                rotation=90)

    if not (np.isnan(no_demo) or np.isnan(with_demo)):
        delta = with_demo - no_demo
        top = max(no_demo, with_demo) + 0.012
        if delta > 0:
            color = COLOR_DELTA_POS
        elif delta < 0:
            color = COLOR_DELTA_NEG
        else:
            color = "0.3"
        ax.text(x_center, top, f"{delta:+.3f}",
                ha="center", va="bottom", fontsize=8.5, color=color,
                fontweight="bold")


def make_figure(out_path=None):
    fig, axes = plt.subplots(len(METRICS), len(DISEASES),
                              figsize=(9, 5.5), sharey=False)

    x_pos = np.arange(len(METHODS))

    for row, metric in enumerate(METRICS):
        for col, disease in enumerate(DISEASES):
            ax = axes[row, col]

            for i, method in enumerate(METHODS):
                no_demo, with_demo = RESULTS[(disease, method, metric)]
                _draw_pair(ax, x_pos[i], no_demo, with_demo)

            demo_y = DEMO_ONLY[(disease, metric)]
            if not np.isnan(demo_y):
                ax.axhline(demo_y, color=COLOR_DEMO_LINE,
                           linestyle="--", linewidth=1.5, zorder=2)

            ax.set_xticks(x_pos)
            if row == len(METRICS) - 1:
                ax.set_xticklabels(METHODS, rotation=20, ha="right",
                                   fontsize=9)
            else:
                ax.set_xticklabels([])
            if row == 0:
                ax.set_title(disease, fontsize=12, fontweight="bold")
            ax.set_ylim(Y_LO, Y_HI)
            ax.grid(axis="y", linestyle=":", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            if col == 0:
                ax.set_ylabel(metric, fontsize=12, fontweight="bold")

    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLOR_NO_DEMO,
                      edgecolor="black", linewidth=0.6,
                      label="Without demographics"),
        plt.Rectangle((0, 0), 1, 1, facecolor=COLOR_WITH_DEMO,
                      edgecolor="black", linewidth=0.6,
                      label="With demographics added"),
        plt.Line2D([0], [0], color=COLOR_DEMO_LINE, linestyle="--",
                   linewidth=1.5, label="Demographics only"),
    ]
    fig.tight_layout()
    fig.subplots_adjust(top=0.88)
    fig.legend(handles=legend_handles, loc="upper center", ncol=3,
               bbox_to_anchor=(0.5, 0.98), frameon=False, fontsize=9)
    if out_path:
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
    return fig

## Render

In [ ]:
fig = make_figure(out_path="demographic_features_comparison.pdf")
plt.show()